In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

#### Recreating the cifar10 dataset images stats mean and std

In [28]:
from pathlib import Path
import numpy as np
from PIL import Image

PATH = Path("../data/cifar10/train/")

# Get all image paths
image_paths = list(PATH.rglob('*.png')) + list(PATH.rglob('*.jpg'))
print(f"Found {len(image_paths)} images")

# Load all images into a single array (vectorized)
images = np.array([np.array(Image.open(p)) for p in image_paths]) / 255.0
# Shape: (num_images, height, width, channels)

# Calculate mean and std across all pixels (vectorized)
MEAN_cf10, std_cf10 = images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))  # Average & std across images, height, width

print(f"\nMean: {MEAN_cf10}, Std:  {std_cf10}")
print(f"\nstats = (np.array({MEAN_cf10.tolist()}), np.array({std_cf10.tolist()}))")

Found 50000 images

Mean: [0.4914  0.48216 0.44653], Std:  [0.24703 0.24349 0.26159]

stats = (np.array([0.49139967861961836, 0.4821584084000747, 0.4465309144581913]), np.array([0.247032232450512, 0.24348512800087502, 0.26158784173101723]))


In [1]:
from fastai.conv_learner import *
PATH = Path("../data/cifar10/")
os.makedirs(PATH, exist_ok=True)

In [2]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(device)

In [3]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
#stats = (np.array([ 0.4914, 0.48216, 0.44653]), np.array([ 0.24703, 0.24349, 0.26159]))

stats = (np.array([0.49139967861961836, 0.4821584084000747, 0.4465309144581913]), 
         np.array([0.247032232450512, 0.24348512800087502, 0.26158784173101723]))

num_workers = num_cpus()//2
bs=256
sz=32

In [4]:
#after padding it becomes 36x36 image, but then it randomly crops and take 32x32 image
tfms = tfms_from_stats(stats, sz, aug_tfms=[RandomFlip()], pad=sz//8) 
data = ImageClassifierData.from_paths(PATH, val_name='test', tfms=tfms, bs=bs)

flipping and reverse padding instead of black pads, resulting better results and used in fastai
- pad = x[..., -n:].flip(-1)
- x = torch.cat([x, pad], dim=-1)

In [5]:
def conv_layer(ni, nf, ks=3, stride=1):
    """ bias is False, coz BatchNorm has learnable bias, which will cancel/neuralize the bias even if used 
     In BatchNorm Normalization: x̂ = (x - μ) / √(σ² + ε) ....scale and shift: y = γx̂ + β (gamma and beta are learnable parameters)."""
    return nn.Sequential(
        nn.Conv2d(ni, nf, kernel_size=ks, bias=False, stride=stride, padding=ks//2),#calculate padding on the go instead of having it as parameter
        nn.BatchNorm2d(nf, momentum=0.01),
        nn.LeakyReLU(negative_slope=0.1, inplace=True)) #memory saving trick, modify input tensor directly instead of creating a new tensor for output, safe for autograd since its the last operation on the block, activation output is not reused

In [6]:
class ResLayer(nn.Module):
    def __init__(self, ni):
        """ 1x1 conv kernel magic, better memory and spatial  structure preservation, also allow change in channel dimention to whatever we want.
        even used in - `Vision Transformers sometimes use conv stems for initial patch embedding` """
        super().__init__()
        self.conv1=conv_layer(ni, ni//2, ks=1) #bottleneck reduction
        self.conv2=conv_layer(ni//2, ni, ks=3)
    def forward(self, x): 
        #return x.add_(self.conv2(self.conv1(x))) #inplace operation, might cause autograd computing gradients issue
        return self.conv2(self.conv1(x)) + x

In [7]:
class Darknet(nn.Module):
    def make_group_layer(self, ch_in, num_blocks, stride=1):
        return [conv_layer(ch_in, ch_in*2, stride=stride)
               ] + [(ResLayer(ch_in*2)) for i in range(num_blocks)] # conv_layer ->ResLayer+ N*(ResLayer)
    
    def __init__(self, num_blocks, num_classes, nf=32):
        super().__init__()
        layers = [conv_layer(3, nf, ks=3, stride=1)]
        for i, nb in enumerate(num_blocks):
            layers += self.make_group_layer(nf, nb, stride=2-(i==1))
            nf *= 2
        layers += [nn.AdaptiveAvgPool2d(1), Flatten(), nn.Linear(nf, num_classes)]
        self.layers = nn.Sequential(*layers)
    
    def forward(self, x): return self.layers(x)

In [14]:
# Fix 2: Disable cuDNN benchmark
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [15]:
m = Darknet([1, 2, 4, 6, 3], num_classes=10, nf=32) #this will be 32 conv layers, 224x224->14x14 before pooling
#m = nn.DataParallel(m, device_ids=None)

In [16]:
lr = 1.3

In [ ]:
#not needed, the print patch fixes and go back to fastai accuracy 

# def my_metric(preds, targets):
#     _,argmax = torch.max(preds, dim=1)
#     acc = argmax.eq(targets).float().mean()
#     if isinstance(acc, float): return acc
#     if hasattr(acc, 'item'): return acc.item() 
#     try: return acc[0]
#     except: return float(acc)

In [41]:
learn = ConvLearner.from_model_data(m, data)
learn.crit = nn.CrossEntropyLoss()
learn.metrics = [accuracy]#[my_metric]
wd=1e-4

In [42]:
import fastai.model
import numpy as np

# 1. Fix the printing crash
def fixed_print_stats(epoch, values, decimals=6):
    layout = "{!s:^10}" + " {!s:10}" * len(values)
    # Convert each item to float and round it individually
    clean_values = [round(float(v), decimals) for v in values]
    print(layout.format(*( [epoch] + clean_values )))

# 2. Fix the history saving crash (your current error)
def fixed_append_stats(ep_vals, epoch, values, decimals=6):
    # Convert each item to float and round it individually
    ep_vals[epoch] = [round(float(v), decimals) for v in values]
    return ep_vals

# Overwrite both library functions with the fixed versions
fastai.model.print_stats = fixed_print_stats
fastai.model.append_stats = fixed_append_stats

print("✅ Patches applied: print_stats and append_stats are now NumPy 1.16+ compatible.")

✅ Patches applied: print_stats and append_stats are now NumPy 1.16+ compatible.


In [ ]:
%time learn.fit(lr, 1, wds=wd, cycle_len=30, use_clr_beta=(20, 20, 0.95, 0.85))

Epoch:   0%|          | 0/30 [00:00<?, ?it/s]

epoch      trn_loss   val_loss   accuracy                   
    0      1.008699   1.961795   0.4118    
    1      0.986122   1.53504    0.5164                      
    2      0.908944   1.799523   0.452                       
    3      0.853318   1.021388   0.6489                      
    4      0.791265   0.966382   0.6664                      
    5      0.73147    1.192296   0.6056                      
    6      0.677184   1.059745   0.6225                      
    7      0.651474   1.04134    0.6532                      
    8      0.634303   0.911548   0.6934                      
    9      0.613267   1.046994   0.659                       
    10     0.612205   0.748533   0.7443                      
    11     0.588054   0.950369   0.6812                      
    12     0.557057   1.387193   0.5621                      
    13     0.542884   0.973134   0.6852                      
 51%|█████     | 99/196 [00:41<00:41,  2.31it/s, loss=0.528]

In [ ]:
#DP: m = WideResNet(depth=22, num_classes=10, widen_factor=6,dropRate=0.)
%time learn.fit(lr/10, 1, wds=wd, cycle_len=40, use_clr_beta=(100,1,0.9,0.8))

In [ ]:
%time learn.fit(lr/10, 1, wds=wd, cycle_len=40, use_clr_beta=(20, 20, 0.95, 0.85))

In [ ]:
%time learn.fit(lr, 1, wds=wd, cycle_len=1, use_clr_beta=(100,1,0.9,0.8))

In [ ]:
%time learn.fit(lr, 1, wds=wd, cycle_len=40, use_clr_beta=(10, 15, 0.95, 0.85))

In [ ]:
%time learn.fit(lr/10, 1, wds=wd, cycle_len=1, use_clr_beta=(100,1,0.9,0.8))

In [ ]:
%time learn.fit(1., 1, wds=wd, cycle_len=30, use_clr_beta=(10, 25, 0.95, 0.85))

In [ ]:
%time learn.fit(lr, 1, wds=wd, cycle_len=40, use_clr_beta=(100, 15, 0.95, 0.85))

In [ ]:
#darknet 2222 lr 1.3 65 cl
%time learn.fit(lr, 1, wds=wd, cycle_len=65, use_clr_beta=(30, 20, 0.95, 0.85))